<a href="https://colab.research.google.com/github/gustkt/GE439_AI_for_Social_Good/blob/main/6606614615_Lab_4_Urban_Green_Impervious.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: พื้นที่สีเขียวและพื้นผิวดาดแข็งในเขตเมือง

**เป้าหมาย:** วัดพื้นที่สีเขียว (Urban Green) และพื้นผิวดาดแข็ง (Impervious Surface) ของเมืองที่นักศึกษาเลือก
แล้วแปลงเป็นตัวชี้วัดที่นักผังเมืองใช้จริง: % ดาดแข็ง และพื้นที่เขียวต่อหัวประชากร

แนวคิด: เขียวกับดาดแข็งคือเหรียญเดียวกันคนละด้าน — พิกเซลในเมืองที่ไม่เขียวและไม่ใช่น้ำ ส่วนใหญ่คือคอนกรีต


In [1]:
import ee
import geemap
ee.Authenticate()
# เปลี่ยนเป็น Cloud Project ของนักศึกษาเอง
ee.Initialize(project='capstone-491411')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [3]:
# ---- 1) เลือกพื้นที่ศึกษา: อำเภอ/เขต จาก FAO GAUL level 2 (public) ----
PROVINCE = 'Pathum Thani'          # จังหวัด (สะกดอังกฤษ)
DISTRICT = 'Thanyaburi'   # อำเภอ — ลองเปลี่ยนเป็นเมืองของนักศึกษาเอง

aoi = (ee.FeatureCollection('FAO/GAUL/2015/level2')
       .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))
       .filter(ee.Filter.eq('ADM1_NAME', PROVINCE))
       .filter(ee.Filter.eq('ADM2_NAME', DISTRICT)))
n = aoi.size().getInfo()
print('features:', n)
if n == 0:
    # ถ้าหาไม่เจอ ให้ list ชื่ออำเภอทั้งหมดของจังหวัดมาดูการสะกด
    names = (ee.FeatureCollection('FAO/GAUL/2015/level2')
             .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))
             .filter(ee.Filter.eq('ADM1_NAME', PROVINCE))
             .aggregate_array('ADM2_NAME').getInfo())
    print('อำเภอที่มีในจังหวัดนี้:', names)

features: 1


In [4]:
# ---- 2) ภาพ Sentinel-2 หน้าแล้ง (เมฆน้อย เห็นเมืองชัด) ----
def mask_s2_clouds(img):
    scl = img.select('SCL')
    good = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)))
    return img.updateMask(good)

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi.geometry())
      .filterDate('2026-01-01', '2026-04-30')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
      .map(mask_s2_clouds)
      .median().clip(aoi))
print('ok')

ok


In [5]:
# ---- 3) คำนวณ EVI แล้วแยก เขียว / น้ำ / ดาดแข็ง ----
evi = s2.expression(
    '2.5 * (NIR - RED) / (NIR + 6*RED - 7.5*BLUE + 1)',
    {'NIR': s2.select('B8').divide(10000),
     'RED': s2.select('B4').divide(10000),
     'BLUE': s2.select('B2').divide(10000)}).rename('EVI')

ndwi = s2.normalizedDifference(['B3', 'B8']).rename('NDWI')   # น้ำ

EVI_THRESHOLD = 0.4     # ลองปรับ แล้วดูผลในข้อ 6
water  = ndwi.gt(0.1)
green  = evi.gte(EVI_THRESHOLD).And(water.Not()).selfMask()
imperv = evi.lt(EVI_THRESHOLD).And(water.Not()).selfMask()

In [6]:
# ---- 4) แผนที่ ----
Map = geemap.Map()
Map.add_basemap('HYBRID')
Map.addLayer(evi, {'min': 0, 'max': 0.8, 'palette': ['white', 'yellow', 'green']}, 'EVI', False)
Map.addLayer(green,  {'palette': ['#1a9641']}, 'พื้นที่สีเขียว')
Map.addLayer(imperv, {'palette': ['#d7191c']}, 'พื้นผิวดาดแข็ง')
Map.addLayer(water.selfMask(), {'palette': ['#2c7fb8']}, 'น้ำ')
Map.centerObject(aoi, 12)
Map

Map(center=[14.02770107599455, 100.76466201168894], controls=(WidgetControl(options=['position', 'transparent_…

In [7]:
# ---- 5) ตัวชี้วัด ----
def area_km2(mask_img):
    a = (mask_img.multiply(ee.Image.pixelArea())
         .reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi.geometry(),
                       scale=20, maxPixels=1e10, bestEffort=True))
    return ee.Number(a.values().get(0)).divide(1e6).getInfo()

total_km2  = aoi.geometry().area().divide(1e6).getInfo()
green_km2  = area_km2(green.unmask(0))
imperv_km2 = area_km2(imperv.unmask(0))

POPULATION = 219731   # <<-- แก้เป็นประชากรจริงของเขตที่เลือก (ข้อมูลจาก ระบบสถิติทางการทะเบียน สำนักบริหารการทะเบียน กรมการปกครอง เดือนธันวาคม 2568)

print(f'พื้นที่ทั้งหมด        {total_km2:,.1f} ตร.กม.')
print(f'พื้นที่สีเขียว        {green_km2:,.1f} ตร.กม. ({green_km2/total_km2*100:.1f}%)')
print(f'พื้นผิวดาดแข็ง       {imperv_km2:,.1f} ตร.กม. ({imperv_km2/total_km2*100:.1f}%)')
print(f'พื้นที่เขียวต่อหัว    {green_km2*1e6/POPULATION:,.1f} ตร.ม./คน (เกณฑ์แนะนำ WHO ~9 ตร.ม./คน)')

พื้นที่ทั้งหมด        125.4 ตร.กม.
พื้นที่สีเขียว        35.5 ตร.กม. (28.3%)
พื้นผิวดาดแข็ง       87.2 ตร.กม. (69.6%)
พื้นที่เขียวต่อหัว    161.6 ตร.ม./คน (เกณฑ์แนะนำ WHO ~9 ตร.ม./คน)


## 6) การทดลองความไว (Sensitivity)

กลับไปแก้ `EVI_THRESHOLD` เป็น 0.2 และ 0.4 แล้วรันซ้ำ บันทึกตัวเลขทั้ง 3 กรณีลงตาราง:

| EVI threshold | % เขียว | % ดาดแข็ง | เขียวต่อหัว (ตร.ม./คน) |
|---|---|---|---|
| 0.2 | 67.3%  | 30.6% | 384.2 |
| 0.3 | 48.8%  | 49.1% | 278.3 |
| 0.4 | 28.3% | 69.6%  | 161.6 |


# คำถามท้ายบทปฏิบัติการ

1. จากตาราง sensitivity — ตัวเลข "พื้นที่เขียวต่อหัว" แกว่งแค่ไหนเมื่อขยับ threshold? ถ้าต้องรายงานตัวเลขเดียวต่อสาธารณะ จะเลือกค่าไหนและจะเขียนหมายเหตุกำกับว่าอย่างไร

2. วิธีนี้นับ "สนามหญ้าในบ้านจัดสรร" กับ "นาข้าวชานเมือง" เป็นพื้นที่เขียวด้วย — สิ่งนี้ทำให้ตัวชี้วัด "เขียวต่อหัว" สูงเกินความหมายเชิงนโยบายอย่างไร และจะนิยาม "พื้นที่เขียวที่ประชาชนเข้าถึงได้จริง" ให้วัดได้ด้วยข้อมูลอะไรเพิ่ม

3. ถ้าเป็นนายกเทศมนตรีของเมืองที่เลือก จะเพิ่มพื้นที่เขียวที่โซนไหนก่อน — ตอบโดยชี้จากแผนที่ (แคปหน้าจอประกอบ) พร้อมเหตุผล 3-4 บรรทัด

*ส่ง: ลิงก์ Colab + ตาราง sensitivity + คำตอบ 3 ข้อ ใน Google Classroom*
